# ITASORL - end-to-end experiments on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/iLevyTate/ITASORL/blob/main/notebooks/colab_gpu.ipynb)

Runs pytest + every experiment via `scripts/run_e2e.py`, records each run under
`fullruns/<MMDDYYYY>/`, mirrors to Google Drive, and saves a portable `bundle.zip`.

**Research question.** Does a from-scratch survival agent *incidentally* encode which
world it lives in (authentic vs a surrogate whose drag differs), read out by a linear
probe but never rewarded? The pre-registered bar is **probe AUROC >= 0.65**.

### How to use
1. **File -> Save a copy in Drive** (the first cell blocks the read-only GitHub copy).
2. **Runtime -> Change runtime type -> GPU** (T4 / L4 / A100).
3. Pick a `RUN_PROFILE` in the config form (dropdown, no code edits).
4. **Runtime -> Run all**. Read the printed `SUMMARY.md` and the variance /
   selectivity re-analysis at the bottom when it finishes.

### Run profiles
| Profile | ~Time | What it runs |
|---------|-------|--------------|
| `quick` | ~25 min | smoke test (reduced scale) |
| `full` | ~4 hr | full replication: A + B + B-v2 (300 upd, 3 seeds) |
| `bv3_regime` | ~3.7 hr | **B-v3 threshold attempt** - per-episode drag *regime* (identifiable + policy-relevant) |
| `bv3_regime_n10` | ~11.5 hr | **B-v3 power extension** - fresh seeds 0-9, regime coupling; resume across sessions |
| `bv2_ceiling` | ~3.7 hr | B-v2 (ar1) capacity ceiling (`--sysid-aux`): what the trunk *can* encode |
| `bv3_ceiling` | ~3.7 hr | B-v3 capacity ceiling (`--drift-mode regime --sysid-aux`): is the 0.65 bar reachable |
| `b2_only` | ~3.7 hr | B-v2 only, 3 seeds |
| `b2_seed0` | ~75 min | B-v2 only, seed 0 (replication-gap diagnostic) |
| `experiments_no_b2` | ~15 min | everything except B-v2 |

All B-v2 / B-v3 profiles **dump recurrent states** so the variance-signature and
selectivity probes can be recomputed offline (last cell) with no GPU.

> **If Colab disconnects** (free tier caps GPU sessions at ~90 min): reopen
> your Drive copy and **Runtime -> Run all** again. The notebook finds the
> unfinished run on Drive and resumes it automatically; checkpoints mirror to
> Drive every ~5 minutes during the run, so at most a few minutes repeat.

In [ ]:
# @title 0. Copy to Drive to keep your edits (Colab)
# Opening this notebook from GitHub shows a read-only playground copy where
# notebook edits won't save. The run still works, but for anything you want
# to keep, File -> Save a copy in Drive first.
try:
    import google.colab  # noqa: F401
    _in_colab = True
except ImportError:
    _in_colab = False

if _in_colab:
    try:
        from google.colab.output import eval_js
        _nb_url = eval_js('window.location.href')
    except Exception:
        _nb_url = ''
    if 'colab.research.google.com/github/' in _nb_url:
        print('!' * 72)
        print('WARNING: read-only GitHub copy detected. Notebook edits will not')
        print('save. File -> Save a copy in Drive to keep them. Continuing.')
        print('!' * 72)
    else:
        print('OK: Colab environment ready.')
else:
    print('Local mode: guard not needed.')

## 1. Configure the run

Pick a `RUN_PROFILE` from the dropdown. `FORCE_FRESH` abandons an unfinished
Drive run and starts over. `RESUME_RUN_DIR` is an advanced override (leave
empty for automatic resume). `BRANCH` other than `main` is for testing
unmerged code only.

In [ ]:
# @title Configure { display-mode: "both" }
from pathlib import Path

RUN_PROFILE = "bv3_regime"  # @param ["quick", "full", "bv3_regime", "bv3_regime_n10", "bv2_ceiling", "bv3_ceiling", "b2_only", "b2_seed0", "experiments_no_b2"]
BRANCH = "main"  # @param {type:"string"}
FORCE_FRESH = False  # @param {type:"boolean"}
RESUME_RUN_DIR = ""  # @param {type:"string"}

REPO_URL = "https://github.com/iLevyTate/ITASORL.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
    REPO_DIR = "/content/ITASORL"
except ImportError:
    IN_COLAB = False
    REPO_DIR = str(Path.cwd().resolve())
    if not (Path(REPO_DIR) / "scripts" / "run_e2e.py").is_file():
        REPO_DIR = str(Path(REPO_DIR).parent)

# Keep _PROFILES in sync with PROFILES in scripts/run_local.py.
# only="expb2" runs B-v2/B-v3 alone (run_e2e.py skips pytest + Experiment A + B).
_PROFILES = {
    "quick":             dict(run_mode="quick", only=None,    skip_steps=[],        b2_seeds=None, b2_updates=None, drift_mode=None,     sysid_aux=False, dump_states=True),
    "full":              dict(run_mode="full",  only=None,    skip_steps=[],        b2_seeds=None, b2_updates=None, drift_mode=None,     sysid_aux=False, dump_states=True),
    "bv3_regime":        dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=None, b2_updates=300,  drift_mode="regime", sysid_aux=False, dump_states=True),
    "bv3_regime_n10":    dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=list(range(10)), b2_updates=300, drift_mode="regime", sysid_aux=False, dump_states=True),
    "bv2_ceiling":       dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=None, b2_updates=300,  drift_mode=None,     sysid_aux=True,  dump_states=True),
    "bv3_ceiling":       dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=None, b2_updates=300,  drift_mode="regime", sysid_aux=True,  dump_states=True),
    "b2_only":           dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=None, b2_updates=300,  drift_mode=None,     sysid_aux=False, dump_states=True),
    "b2_seed0":          dict(run_mode="full",  only="expb2", skip_steps=[],        b2_seeds=[0],  b2_updates=300,  drift_mode=None,     sysid_aux=False, dump_states=True),
    "experiments_no_b2": dict(run_mode="full",  only=None,    skip_steps=["expB2"], b2_seeds=None, b2_updates=None, drift_mode=None,     sysid_aux=False, dump_states=False),
}
if RUN_PROFILE not in _PROFILES:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose from {list(_PROFILES)}")
_p = _PROFILES[RUN_PROFILE]
RUN_MODE = _p["run_mode"]
E2E_ONLY = _p["only"]                 # None | "pytest" | "experiments" | "expb2"
SKIP_STEPS = list(_p["skip_steps"])
B2_SEEDS = _p["b2_seeds"]
B2_UPDATES = _p["b2_updates"]
DRIFT_MODE = _p["drift_mode"]        # None | "ar1" | "regime" (B-v3)
SYSID_AUX = _p["sysid_aux"]          # capacity-ceiling control (breaks readout-not-reward)
DUMP_STATES = _p["dump_states"]      # persist recurrent states for offline re-probing

MOUNT_DRIVE = IN_COLAB
DRIVE_RESULTS = "/content/drive/MyDrive/ITASORL_results"
SAVE_TO_DRIVE = IN_COLAB
DOWNLOAD_BUNDLE = IN_COLAB
RESUME_RUN_DIR = RESUME_RUN_DIR.strip()

if BRANCH != "main":
    print("!" * 72)
    print(f"WARNING: running branch {BRANCH!r}, not main. For testing unmerged")
    print("code only; do not treat the results as replication runs.")
    print("!" * 72)

print(f"RUN_PROFILE={RUN_PROFILE}  RUN_MODE={RUN_MODE}  only={E2E_ONLY}  skip={SKIP_STEPS or '(none)'}")
print(f"B-v2/3: drift_mode={DRIFT_MODE}  sysid_aux={SYSID_AUX}  dump_states={DUMP_STATES}"
      + (f"  seeds={B2_SEEDS}" if B2_SEEDS is not None else "")
      + (f"  updates={B2_UPDATES}" if B2_UPDATES is not None else ""))
if FORCE_FRESH:
    print("FORCE_FRESH: any unfinished Drive run will be ignored, not resumed.")

## 2. Mount Google Drive

Colab only. Runs are mirrored here so a timeout never loses progress.

In [ ]:
import os, subprocess, sys, shutil, time
from pathlib import Path

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=check)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
elif not IN_COLAB:
    print(f"Local mode (repo: {REPO_DIR})")

## 3. Fetch the code

Clone / update the repo at `BRANCH` and print the exact commit that will run.

In [ ]:
if IN_COLAB:
    if os.path.isdir(REPO_DIR):
        sh(f"cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}")
    else:
        sh(f"git clone --branch {BRANCH} --single-branch {REPO_URL} {REPO_DIR}")
else:
    print("Local mode: using existing checkout (skip clone)")
    if not (Path(REPO_DIR) / "itasorl").is_dir():
        raise FileNotFoundError(f"Expected itasorl/ under {REPO_DIR}; set REPO_DIR in config cell")

os.chdir(REPO_DIR)
sh("git log -1 --oneline")
print("REPO_DIR:", REPO_DIR)

## 4. Install dependencies

In [ ]:
sh(f"{sys.executable} -m pip install -q --upgrade pip")
dev = os.path.join(REPO_DIR, "requirements-dev.txt")
if os.path.isfile(dev):
    sh(f"{sys.executable} -m pip install -q -r requirements-dev.txt")
else:
    sh(f"{sys.executable} -m pip install -q -r requirements.txt pytest")

## 5. Check the GPU

Confirm a CUDA device is attached (Runtime -> Change runtime type -> GPU).

In [ ]:
import torch

# Hardware is your choice: Runtime > Change runtime type (not hardcoded to T4).
if torch.cuda.is_available():
    print("CUDA: True")
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("CUDA: False")
    print("No GPU detected. Runtime > Change runtime type > pick a GPU, then re-run this cell.")
    print("CPU-only works but expB2 will be much slower.")

## 6. Enable keep-alive

Colab only. Keeps the session awake during the long run; leave the tab open.

In [ ]:
# Keep-alive (Colab only): run immediately before the long run cell.
if IN_COLAB:
    from IPython.display import display, Javascript

    display(Javascript("""
    (function () {
      const intervalMs = 60000;
      setInterval(() => {
        fetch('/gen_' + Date.now()).catch(() => {});
      }, intervalMs);
      console.log('ITASORL keep-alive active every', intervalMs / 1000, 's');
    })();
    """))
    print("Keep-alive enabled. Leave this tab open until the run finishes.")
else:
    print("Local mode: keep-alive not needed.")

## 7. Run the experiments

Auto-resume: if an unfinished run for this profile exists on Drive, it is
copied local and continued; otherwise a fresh run starts. Checkpoints mirror
to Drive every ~5 minutes during the run. The printed `$ ...` line shows the
exact flags.

In [ ]:
import argparse
import json

if str(Path(REPO_DIR) / "scripts") not in sys.path:
    sys.path.insert(0, str(Path(REPO_DIR) / "scripts"))
import run_e2e as _run_e2e  # single source of truth for b2 flag building


def _expected_b2_flags():
    ns = argparse.Namespace(
        b2_seeds=B2_SEEDS, b2_updates=B2_UPDATES, b2_hidden=None,
        b2_dump_states=("auto" if DUMP_STATES else None),
        b2_sysid_aux=SYSID_AUX, b2_drift_mode=DRIFT_MODE)
    return _run_e2e.build_b2_extra(ns)


def _find_unfinished_drive_run():
    root = Path(DRIVE_RESULTS)
    if not root.is_dir():
        return None
    cands = []
    for mf in sorted(root.glob("*/manifest.json")):
        folder = mf.parent
        if (folder / "bundle.zip").is_file():
            continue  # bundle exists only after a successful finish
        try:
            manifest = json.loads(mf.read_text(encoding="utf-8"))
        except (OSError, ValueError) as exc:
            print(f"auto-resume: skipping {folder.name} (unreadable manifest: {exc})", flush=True)
            continue
        cands.append((manifest.get("started_at_utc", ""), folder, manifest))
    if not cands:
        return None
    cands.sort(key=lambda t: t[0])
    _, folder, manifest = cands[-1]
    return folder, manifest


resume_src = None
if RESUME_RUN_DIR:
    resume_src = Path(RESUME_RUN_DIR)
    if not resume_src.is_dir():
        raise FileNotFoundError(f"RESUME_RUN_DIR not found: {resume_src}")
    print("Explicit resume override:", resume_src, flush=True)
elif not FORCE_FRESH and IN_COLAB and MOUNT_DRIVE:
    _found = _find_unfinished_drive_run()
    if _found:
        resume_src, _manifest = _found
        _recorded = []
        _ff = resume_src / "b2_flags.json"
        if _ff.is_file():
            _recorded = json.loads(_ff.read_text(encoding="utf-8"))
        _expected = _expected_b2_flags()
        _quick_ok = bool(_manifest.get("quick", False)) == (RUN_MODE == "quick")
        if _recorded != _expected or not _quick_ok:
            print("!" * 72, flush=True)
            print(f"Skipping auto-resume: {resume_src.name!r} was started with a", flush=True)
            print("different profile.", flush=True)
            print(f"  recorded: {_recorded} (quick={_manifest.get('quick')})", flush=True)
            print(f"  profile {RUN_PROFILE!r} would use: {_expected} (quick={RUN_MODE == 'quick'})", flush=True)
            print(f"Starting a fresh run for {RUN_PROFILE!r} instead.", flush=True)
            print("!" * 72, flush=True)
            resume_src = None  # fall through to fresh run
        else:
            print(f"Auto-resume: unfinished Drive run {resume_src.name!r} matches profile {RUN_PROFILE!r}.",
                  flush=True)

run_dir = None
if resume_src is not None:
    if str(resume_src).startswith("/content/drive"):
        run_dir = Path(REPO_DIR) / "fullruns" / "_resume_local" / resume_src.name
        print("Copying Drive run to local disk for resume:", run_dir, flush=True)
        shutil.copytree(resume_src, run_dir, dirs_exist_ok=True)
    else:
        run_dir = resume_src

cmd = [sys.executable, "scripts/run_e2e.py"]
if RUN_MODE == "quick":
    cmd.append("--quick")
elif RUN_MODE != "full":
    raise ValueError("RUN_MODE must be 'quick' or 'full'")
if E2E_ONLY:
    cmd.extend(["--only", E2E_ONLY])
for step in SKIP_STEPS:
    cmd.extend(["--skip", step])
if run_dir is not None:
    cmd.extend(["--resume", str(run_dir)])
    print("Resuming run (recorded --b2-* flags replay automatically):", run_dir, flush=True)
else:
    if B2_SEEDS is not None:
        cmd.extend(["--b2-seeds", *[str(s) for s in B2_SEEDS]])
    if B2_UPDATES is not None:
        cmd.extend(["--b2-updates", str(B2_UPDATES)])
    if DRIFT_MODE is not None:
        cmd.extend(["--b2-drift-mode", DRIFT_MODE])   # B-v3 regime coupling
    if SYSID_AUX:
        cmd.append("--b2-sysid-aux")                  # capacity-ceiling control
    if DUMP_STATES:
        cmd.extend(["--b2-dump-states", "auto"])      # lands in <run_dir>/artifacts/states
    print("New run; results land under fullruns/ and mirror to Drive.", flush=True)

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"
if MOUNT_DRIVE:
    run_env["ITASORL_DRIVE_SYNC"] = DRIVE_RESULTS
    print("Live mirror -> Drive:", DRIVE_RESULTS, flush=True)


def _current_run_dir():
    if run_dir is not None:
        return Path(run_dir)
    ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
    return Path(ptr.read_text().strip()) if ptr.is_file() else None


def _print_failure_diagnostics():
    rd = _current_run_dir()
    if rd is None or not rd.is_dir():
        print("No run directory found for diagnostics.", flush=True)
        return
    print("\n" + "=" * 72, flush=True)
    print("RUN FAILED. Diagnostics for:", rd, flush=True)
    mf = rd / "manifest.json"
    if mf.is_file():
        steps = json.loads(mf.read_text()).get("steps", {})
        failed = [n for n, s in steps.items() if s.get("status") == "failed"]
        ok = [n for n, s in steps.items() if s.get("status") == "ok"]
        if ok:
            print("Completed steps:", ", ".join(ok), flush=True)
        if failed:
            print("Failed steps:", ", ".join(failed), flush=True)
            for name in failed:
                log = rd / "steps" / f"{name}.log"
                if log.is_file():
                    tail = log.read_text(errors="replace").splitlines()[-40:]
                    print(f"\n--- tail {name}.log ---", flush=True)
                    print("\n".join(tail), flush=True)
    cl = rd / "combined.log"
    if cl.is_file():
        tail = cl.read_text(errors="replace").splitlines()[-30:]
        print("\n--- tail combined.log ---", flush=True)
        print("\n".join(tail), flush=True)
    print(f"Everything so far is mirrored on Drive: {DRIVE_RESULTS}/{rd.name}", flush=True)
    print("To pick up where this left off: reopen your Drive copy of this", flush=True)
    print("notebook and Runtime -> Run all (auto-resume finds this run).", flush=True)
    print("=" * 72, flush=True)


print("$", " ".join(cmd), flush=True)
t0 = time.perf_counter()
proc = subprocess.run(cmd, cwd=REPO_DIR, env=run_env)
if proc.returncode != 0:
    _print_failure_diagnostics()
    raise subprocess.CalledProcessError(proc.returncode, proc.args)
print(f"Wall time: {(time.perf_counter() - t0) / 60:.1f} min")

## 8. Read the results

Print `SUMMARY.md` and compare B-v2 survival @ drift 0.45 to the canonical artifacts.

In [ ]:
latest_ptr = Path(REPO_DIR) / "results" / "LATEST_RUN.txt"
RUN_DIR = Path(latest_ptr.read_text().strip())
BUNDLE = RUN_DIR / "bundle.zip"
SUMMARY = RUN_DIR / "SUMMARY.md"

print("Run directory:", RUN_DIR)
print("Bundle:", BUNDLE, "exists:", BUNDLE.is_file())
print("\n" + "=" * 72)
print(SUMMARY.read_text())
print("=" * 72)

# B-v2 comparison vs canonical + initial lab artifacts (no GPU), if the script is present.
_cmp = Path(REPO_DIR) / "scripts" / "compare_expB2_artifacts.py"
if _cmp.is_file():
    print("\n" + "=" * 72)
    print("B-v2 artifact comparison (survival @ drift 0.45)")
    print("=" * 72)
    subprocess.run([sys.executable, str(_cmp), "--run", str(RUN_DIR)], cwd=REPO_DIR, check=False)
else:
    print("(compare_expB2_artifacts.py not on this branch; skipping artifact comparison)")

## 9. Variance & selectivity re-analysis (no GPU)

The cheapest test of the central question: is world identity encoded as a *volatility* signature the level probe discards?

In [ ]:
# Variance-signature + selectivity re-analysis (NO GPU). Recomputes the world-identity
# probe under level vs dispersion features from the states dumped during the run:
#   target       LEVEL      [mean h, final h]   (the pre-registered headline)
#   target_var   DISPERSION [std h, mean|delta h|]
#   target_full  LEVEL ++ DISPERSION
# target_var/full crossing 0.65 while level target stays ~0.52 => the null was mis-probed.
STATES = RUN_DIR / "artifacts" / "states"
if DUMP_STATES and STATES.is_dir():
    subprocess.run([sys.executable, "scripts/reanalyze_expB2_states.py", str(STATES)],
                   cwd=REPO_DIR, check=False)
else:
    print("No dumped states found at", STATES)
    print("Use a profile with dump_states=True (all B-v2/B-v3 profiles do) to enable this.")

## 10. Save & download the bundle

In [ ]:
if not BUNDLE.is_file():
    raise FileNotFoundError(f"Missing bundle: {BUNDLE}")

print("Local fullruns run:", RUN_DIR)
print("Local bundle:", BUNDLE)

if SAVE_TO_DRIVE and MOUNT_DRIVE:
    drive_root = Path(DRIVE_RESULTS)
    drive_run = drive_root / RUN_DIR.name
    shutil.copytree(RUN_DIR, drive_run, dirs_exist_ok=True)
    print("Full run copied to Drive:", drive_run)

    drive_bundle = drive_root / f"{RUN_DIR.name}_bundle.zip"
    shutil.copy2(BUNDLE, drive_bundle)
    print("Bundle copied to Drive:", drive_bundle)

    shutil.copy2(SUMMARY, drive_root / f"{RUN_DIR.name}_SUMMARY.md")
    print("SUMMARY copied to Drive")

if DOWNLOAD_BUNDLE:
    from google.colab import files
    files.download(str(BUNDLE))
    print("Browser download started for bundle.zip")
else:
    print("Open SUMMARY:", SUMMARY)

## What's in the bundle

| File | Purpose |
|------|---------|
| `SUMMARY.md` | Plain English outcome. **Read this first.** |
| `status.json` | Live step + last line (updated during run) |
| `manifest.json` | Step timings, status, artifact index |
| `combined.log` | Full stdout (updated live during run) |
| `steps/*.json` | Parsed metrics (AUROC per drift, etc.) |
| `artifacts/` | Figures + `expB2_results.json` |
| `artifacts/cells/` | Per (drift, seed) checkpoints (resume granularity) |
| `artifacts/states/` | Dumped recurrent states for offline re-probing |

**While running (local):** `python scripts/watch_run.py --follow`

**While running (Colab + Drive):** open `MyDrive/ITASORL_results/<run>/combined.log`

**If Colab disconnected:** reopen your Drive copy of this notebook and
**Runtime -> Run all**. Auto-resume finds the newest unfinished run in
`MyDrive/ITASORL_results`, checks it matches your selected profile, and
continues it. Checkpoints mirror to Drive every ~5 minutes during the run.

Unzip locally and open `SUMMARY.md` to decide whether the organism encoded world identity.